In [2]:
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Styling
import matplotlib.style as style

style.use("fivethirtyeight")

# Statistics
from scipy.stats import zscore, norm

# Sklearn - Preprocessing
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    LabelEncoder,
    OneHotEncoder,
)

# Sklearn - Model Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
)

# Sklearn - Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Sklearn - Metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Imbalanced Learning
# from imblearn.over_sampling import SMOTE


# Set display options for better readability

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
np.random.seed(42)

# Define the data file path
DATA_PATH = "student-data-25s3.csv"
MOVIE_LENS_PATH = "ml-100k"

# 1. Data Preparation

Load and inspect the dataset for quality issues.

In [5]:
# Load the data
df = pd.read_csv(DATA_PATH)

# Basic info
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (650, 25)
Columns: ['school', 'sex', 'age', 'address', 'famsize', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'traveltime', 'studytime', 'failures', 'activities', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'health', 'absences', 'G1', 'G2', 'G3']


,school,sex,age,address,famsize,Medu,Fedu,Mjob,Fjob,reason,traveltime,studytime,failures,activities,higher,internet,romantic,famrel,freetime,goout,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,4,4,at_home,teacher,course,2,2,0,no,yes,no,no,4,3,4,3,4,0,11,11
1,GP,F,17,U,GT3,1,1,at_home,other,course,1,2,0,no,yes,yes,no,5,3,3,3,2,9,11,11
2,GP,F,15,U,LE3,1,1,at_home,other,other,1,2,0,no,yes,yes,no,4,3,2,3,6,12,13,12
3,GP,F,15,U,GT3,4,2,health,services,home,1,3,0,yes,yes,yes,yes,3,2,2,5,0,14,14,14
4,GP,F,16,U,GT3,3,3,other,other,home,1,2,0,no,yes,no,no,4,3,2,5,0,11,13,13


## 2.1 Check Duplicates

In [6]:
# Count duplicate rows
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")

# View duplicate rows (if any)
if dup_count > 0:
    display(df[df.duplicated(keep=False)])

# Remove duplicates (uncomment to use)
# df = df.drop_duplicates()
# print(f"Shape after removing duplicates: {df.shape}")

Duplicate rows: 6


,school,sex,age,address,famsize,Medu,Fedu,Mjob,Fjob,reason,traveltime,studytime,failures,activities,higher,internet,romantic,famrel,freetime,goout,health,absences,G1,G2,G3
58,GP,M,15,U,LE3,1,2,other,at_home,home,1,2,0,yes,yes,yes,no,4,3,2,5,0,14,13,14
59,GP,M,15,U,LE3,1,2,other,at_home,home,1,2,0,yes,yes,yes,no,4,3,2,5,0,14,13,14
60,GP,M,15,U,LE3,1,2,other,at_home,home,1,2,0,yes,yes,yes,no,4,3,2,5,0,14,13,14
299,GP,F,20,R,GT3,2,1,other,other,course,2,2,0,yes,no,yes,yes,1,2,3,2,8,10,12,12
300,GP,F,20,R,GT3,2,1,other,other,course,2,2,0,yes,no,yes,yes,1,2,3,2,8,10,12,12
301,GP,F,20,R,GT3,2,1,other,other,course,2,2,0,yes,no,yes,yes,1,2,3,2,8,10,12,12
448,MS,F,16,R,GT3,4,4,teacher,teacher,course,2,3,0,yes,yes,yes,yes,4,2,2,4,6,16,16,17
468,MS,F,16,R,GT3,4,4,teacher,teacher,course,2,3,0,yes,yes,yes,yes,4,2,2,4,6,16,16,17
473,MS,F,16,R,GT3,4,4,teacher,teacher,course,2,3,0,yes,yes,yes,yes,4,2,2,4,6,16,16,17


## 2.2 Check Null Values

In [7]:
# Count null values per column
null_counts = df.isnull().sum()
null_percent = (df.isnull().sum() / len(df) * 100).round(2)

null_summary = pd.DataFrame({
    'null_count': null_counts,
    'null_percent': null_percent
})
print(null_summary[null_summary['null_count'] > 0])

# Total nulls
print(f"\nTotal null values: {df.isnull().sum().sum()}")

           null_count  null_percent
school              1          0.15
sex                 1          0.15
Mjob                1          0.15
Fjob                1          0.15
studytime           3          0.46
G1                  1          0.15

Total null values: 8


## 2.3 Check Unique Values

In [13]:
# Show unique values for each column
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"{col}: {len(unique_vals)} unique values")
    # Show unique values and their counts
    
    for val in unique_vals:
        count = (df[col] == val).sum()
        print(f"  {val}: {count}")
    
    print("-----------")
    

school: 3 unique values
  GP: 423
  MS: 226
  nan: 0
-----------
sex: 3 unique values
  F: 384
  M: 265
  nan: 0
-----------
age: 14 unique values
  18: 138
  17: 178
  15: 111
  16: 173
  19: 32
  22: 1
  20: 8
  21: 2
  105: 1
  106: 1
  115: 1
  2O: 1
  1O: 1
  170: 2
-----------
address: 3 unique values
  U: 449
  R: 200
  0: 1
-----------
famsize: 3 unique values
  GT3: 456
  LE3: 193
  0: 1
-----------
Medu: 5 unique values
  4: 174
  1: 145
  3: 139
  2: 186
  0: 6
-----------
Fedu: 5 unique values
  4: 130
  1: 175
  2: 209
  3: 129
  0: 7
-----------
Mjob: 10 unique values
  at_home: 127
  health: 46
  other: 262
  services: 130
  teacher: 73
  service: 3
  nan: 0
  at home: 4
  home: 3
  0: 1
-----------
Fjob: 8 unique values
  teacher: 37
  other: 360
  services: 179
  health: 23
  at_home: 44
  nan: 0
  others: 5
  0: 1
-----------
reason: 5 unique values
  course: 286
  other: 71
  home: 150
  reputation: 142
  0: 1
-----------
traveltime: 4 unique values
  2: 216
  1: 364

## 2.4 Unique Values Count (Value Counts)

In [ ]:
# Value counts for a specific column (replace 'column_name')
# df['column_name'].value_counts()

# Value counts with percentages
# df['column_name'].value_counts(normalize=True) * 100

# Value counts including NaN
# df['column_name'].value_counts(dropna=False)

## 2.5 Check Outliers

In [15]:
# IQR Method for outlier detection
def detect_outliers_iqr(df, col):
    """Detect outliers using IQR method"""
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    return len(outliers), lower, upper

# Check outliers for all numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
print("Outlier Detection (IQR Method):")
print("-" * 60)
for col in numerical_cols:
    count, lower, upper = detect_outliers_iqr(df, col)
    print(f"{col}: {count} outliers (valid range: {lower:.2f} - {upper:.2f})")

Outlier Detection (IQR Method):
------------------------------------------------------------
Medu: 0 outliers (valid range: -1.00 - 7.00)
Fedu: 0 outliers (valid range: -2.00 - 6.00)
traveltime: 17 outliers (valid range: -0.50 - 3.50)
failures: 101 outliers (valid range: 0.00 - 0.00)
famrel: 51 outliers (valid range: 2.50 - 6.50)
freetime: 44 outliers (valid range: 1.50 - 5.50)
goout: 0 outliers (valid range: -1.00 - 7.00)
health: 0 outliers (valid range: -2.50 - 9.50)
absences: 22 outliers (valid range: -9.00 - 15.00)
G3: 16 outliers (valid range: 4.00 - 20.00)


In [ ]:
# Z-Score Method for outlier detection
def detect_outliers_zscore(df, col, threshold=3):
    """Detect outliers using Z-score method"""
    z_scores = np.abs(zscore(df[col].dropna()))
    outliers = df[col].dropna()[z_scores > threshold]
    return len(outliers)

print("\nOutlier Detection (Z-Score Method, threshold=3):")
print("-" * 60)
for col in numerical_cols:
    count = detect_outliers_zscore(df, col)
    print(f"{col}: {count} outliers")

In [ ]:
# Visualize outliers with box plots
n_cols = 4
n_rows = math.ceil(len(numerical_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
axes = axes.ravel()

for i, col in enumerate(numerical_cols):
    sns.boxplot(data=df, y=col, ax=axes[i])
    axes[i].set_title(col)

# Hide empty subplots
for j in range(len(numerical_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()